In [ ]:
import os

WORKDIR = os.environ["CONTAINER_WORK_DIR"]
os.chdir(WORKDIR)

In [ ]:
from pathlib import Path
from IPython.display import Audio

In [ ]:
from sj_utils.file.yaml import load_yaml
from sj_utils.collection import SafetyDict
from sj_utils.audio import segment_audio
from sj_ai_utils.datasets.esic_v1 import ESICv1Dataset

In [ ]:
from rt_whisper import streamers, saveloaders
from rt_whisper.data import Param, TokenState

In [ ]:
SEED = 42
SAMPLE_RATE = 16000

ESIC_VAL = f"{WORKDIR}/test/performance_test/esic/data/val.json"  # use in val

STORAGE = f"{WORKDIR}/test/.storage/esic/"
HYPERPARAMETER = f"{WORKDIR}/test/optimize/esic/hyperparameters/20250917/3s/step2_3s-96k-cpm/001_0_046.yaml"
LOG_FILE_PATH = f"{WORKDIR}/logs/RTWhisper.log"

In [ ]:
esic_val = Path(ESIC_VAL)

storage = Path(STORAGE)
log = Path(LOG_FILE_PATH)
if not esic_val.exists():
    raise FileNotFoundError(f"Path {esic_val} does not exist.")
if not storage.exists():
    raise FileNotFoundError(f"Path {storage} does not exist.")
if log.exists():
    with log.open("w"): pass
hyperparameter_path = Path(HYPERPARAMETER)

In [ ]:
dataset = ESICv1Dataset.load(esic_val)

In [ ]:
key, audio, text = dataset[1]
key

In [ ]:
Audio(audio, rate=SAMPLE_RATE)

In [ ]:
segments = segment_audio(audio, mean = 48000, std=0, max_div=0)

In [ ]:
hyperparameter = SafetyDict(load_yaml(hyperparameter_path)[1])

In [ ]:
# overlap = hyperparameter["asr"]["max_overlap_duration"]
# saved_path = storage / str(overlap) / key
# token_streamer = saveloaders.get_token_streamer_loader(saved_path, hyperparameter)

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter(hyperparameter=hyperparameter)
# token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter()

In [ ]:
print(text)

In [ ]:
param = Param()
completed = []
index = -1

In [ ]:
from rt_whisper.processors.asr.data import ASRState

index += 1
chunk = segments[index]

param.chunk = chunk
param.language = "en"
ctx:TokenState = token_streamer.process(param, get_context = True)
result = ctx.extract()
completed.extend(result.completed)
param.update(result, update_prompt=True)

print(f"{index}" + "--" * 20)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.completed]
)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.candidate]
)
print([(t.start, t.end, t.text) for t in result.completed_tokens])
print([(t.start, t.end, t.text) for t in result.candidate_tokens])

In [ ]:
start = result.completed[0].tokens[0].start
end = result.completed[0].tokens[-1].end
Audio(audio[start:end], rate=SAMPLE_RATE)

In [ ]:
Audio(result.context_dict[ASRState].chunk, rate=SAMPLE_RATE)

In [ ]:
param = Param()
completed = []
for i, segment in enumerate(segments):
    param.chunk = segment
    param.language = "en"
    ctx:TokenState = token_streamer.process(param, get_context = True)
    result = ctx.extract()
    completed.extend(result.completed)
    param.update(result, update_prompt=True)

completed.extend(result.candidate)

In [ ]:
for s in completed:
    print(s.lang, s.text)